In [24]:
import os
import torch
import torchaudio
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchaudio import transforms, load
from torch.utils.data import DataLoader, Dataset, random_split

In [25]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mmoreaux/environmental-sound-classification-50")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'environmental-sound-classification-50' dataset.
Path to dataset files: /kaggle/input/environmental-sound-classification-50


In [26]:
csv_path = '/kaggle/input/environmental-sound-classification-50/esc50.csv'
data_path = '/kaggle/input/environmental-sound-classification-50/audio/audio'

In [27]:
import pandas as pd
df = pd.read_csv(csv_path)
df.head()

,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


In [28]:
categories = sorted(df['category'].unique())
print(categories)

['airplane', 'breathing', 'brushing_teeth', 'can_opening', 'car_horn', 'cat', 'chainsaw', 'chirping_birds', 'church_bells', 'clapping', 'clock_alarm', 'clock_tick', 'coughing', 'cow', 'crackling_fire', 'crickets', 'crow', 'crying_baby', 'dog', 'door_wood_creaks', 'door_wood_knock', 'drinking_sipping', 'engine', 'fireworks', 'footsteps', 'frog', 'glass_breaking', 'hand_saw', 'helicopter', 'hen', 'insects', 'keyboard_typing', 'laughing', 'mouse_click', 'pig', 'pouring_water', 'rain', 'rooster', 'sea_waves', 'sheep', 'siren', 'sneezing', 'snoring', 'thunderstorm', 'toilet_flush', 'train', 'vacuum_cleaner', 'washing_machine', 'water_drops', 'wind']


In [29]:
len(categories)

50

In [30]:
index_to_label = {words: ind for ind, words in enumerate(categories)}

In [31]:
tranform = transforms.MelSpectrogram(
    sample_rate = 22050,
    n_mels=64
)
max_len = 500

In [32]:
import torchaudio
class Environmental_sound(Dataset):
    def __init__(self, csv_path, data_path, transforms, max_len):
        self.df = pd.read_csv(csv_path)
        self.data_path = data_path
        self.transforms = transforms
        self.max_len = max_len
        self.audios = []

        for file in os.listdir(data_path):
            if file.endswith('.wav'):
                file_path = os.path.join(data_path, file)

                try:
                    category = list(df[df['filename'] == file]['category'])
                    self.audios.append((file_path, category[0]))
                except Exception as e:
                    print(f'Ошибка {e}')

    def __len__(self):
        return len(self.audios)

    def __getitem__(self, ind):
        file_path, category = self.audios[ind]
        waveform, sample_rate = load(file_path)

        if sample_rate != 22050:
            resample = transforms.Resample(orig_freq=sample_rate, new_freq=22050)
            waveform = resample(waveform)

        spectrogram = self.transforms(waveform).squeeze(0)

        if spectrogram.shape[1] > self.max_len:
            spectrogram = spectrogram[:, :self.max_len]

        if spectrogram.shape[1] < self.max_len:
            count_len = self.max_len - spectrogram.shape[1]
            spectrogram = F.pad(spectrogram, (0, count_len))

        return spectrogram, index_to_label[category]

In [33]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [34]:
dataset = Environmental_sound(csv_path, data_path, tranform, max_len)

train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size

train_data, test_data = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

In [35]:
train = DataLoader(train_data, batch_size=32, shuffle=True)
test = DataLoader(test_data, batch_size=32)

In [36]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=50):
        super(SimpleCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)          # (batch, 64, 500) → (batch, 1, 64, 500)
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [37]:
model = SimpleCNN(num_classes=50).to(device)

In [38]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.000117, weight_decay=0.000110)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

In [39]:
!pip install -q torchcodec

In [47]:
epochs=30
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()
    print(f'Эпоха {epoch+1}/{epochs}, Loss: {running_loss/len(train):.4f}')

Эпоха 1/30, Loss: 3.4559
Эпоха 2/30, Loss: 3.3707
Эпоха 3/30, Loss: 3.2979
Эпоха 4/30, Loss: 3.2025
Эпоха 5/30, Loss: 3.1332
Эпоха 6/30, Loss: 3.0538
Эпоха 7/30, Loss: 2.9865
Эпоха 8/30, Loss: 2.9368
Эпоха 9/30, Loss: 2.8863
Эпоха 10/30, Loss: 2.8367
Эпоха 11/30, Loss: 2.7907
Эпоха 12/30, Loss: 2.7530
Эпоха 13/30, Loss: 2.7058
Эпоха 14/30, Loss: 2.6714
Эпоха 15/30, Loss: 2.6381
Эпоха 16/30, Loss: 2.6116
Эпоха 17/30, Loss: 2.5638
Эпоха 18/30, Loss: 2.5508
Эпоха 19/30, Loss: 2.5060
Эпоха 20/30, Loss: 2.4886
Эпоха 21/30, Loss: 2.4729
Эпоха 22/30, Loss: 2.4295
Эпоха 23/30, Loss: 2.3940
Эпоха 24/30, Loss: 2.3873
Эпоха 25/30, Loss: 2.3619
Эпоха 26/30, Loss: 2.3433
Эпоха 27/30, Loss: 2.3175
Эпоха 28/30, Loss: 2.3077
Эпоха 29/30, Loss: 2.2743
Эпоха 30/30, Loss: 2.2579


In [48]:
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy на тесте: {100 * correct / total:.2f}%')

Accuracy на тесте: 6.50%


In [49]:
torch.save(model.state_dict(), 'cifar100_vgg.pth')